# 00 · Download NOAA hourly weather for the top-50 airports
Files: `https://www.ncei.noaa.gov/oa/local-climatological-data/v2/access/<year>/LCD_<STATION>_<year>.csv` (one file per station-year, ~4–7 MB each).
Docs: https://www.ncei.noaa.gov/oa/local-climatological-data/v2/doc/lcdv2_DOCUMENTATION.pdf

**Approach:** match each top-50 airport to the nearest station using coordinates
(OurAirports `airports.csv` for airport lat/lon, NOAA's GHCNh station list for station lat/lon), then **check which candidate files really exist** before downloading.

In [ ]:
import os, time, math, urllib.request, urllib.error
from pathlib import Path
import duckdb, pandas as pd

In [ ]:
# ---- settings -------------------------------------------------------------
PROJECT = Path(os.path.expanduser("~/Desktop/ADSP32030_Scalable/Project"))
FLIGHTS_PQ = PROJECT/"data/flights/parquet"
DATA = PROJECT/"data/weather"
LCD = "https://www.ncei.noaa.gov/oa/local-climatological-data/v2/access/{y}/LCD_{st}_{y}.csv"
STATION_LIST = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh-station-list.csv"
AIRPORTS_CSV = "https://davidmegginson.github.io/ourairports-data/airports.csv"
YEARS = range(2010, 2027)
TOP_N = 50
MAX_KM = 6            # candidate stations within this distance of the airport
PROBE_YEARS = (2010, 2018, 2024)
PAUSE_SECONDS = 1

In [ ]:
RAW, PQ, META = DATA/"raw", DATA/"parquet", DATA/"meta"
for d in (RAW, PQ, META): d.mkdir(parents=True, exist_ok=True)
UA = {"User-Agent": "adsp32030-flight-project/1.0"}

def fetch(url, timeout=120, method="GET", headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers={**UA, **(headers or {})}, method=method), timeout=timeout)

def download(url, dest):
    with fetch(url, 180) as r, open(dest, "wb") as f:
        while chunk := r.read(1 << 20): f.write(chunk)

def exists(url):
    try:
        fetch(url, 30, "HEAD").close(); return True
    except urllib.error.HTTPError as e:
        if e.code in (404, 403): return False
    except Exception: pass
    try:                                    # some servers refuse HEAD: ask for one byte
        fetch(url, 30, "GET", {"Range": "bytes=0-0"}).close(); return True
    except Exception: return False

def km(lat1, lon1, lat2, lon2):
    p = math.pi/180
    a = math.sin((lat2-lat1)*p/2)**2 + math.cos(lat1*p)*math.cos(lat2*p)*math.sin((lon2-lon1)*p/2)**2
    return 12742*math.asin(math.sqrt(a))

## Step 1 · Top-50 airports with coordinates

In [ ]:
con = duckdb.connect()
top = con.execute(f"""SELECT Origin AS iata, count(*) AS flights FROM read_parquet('{FLIGHTS_PQ}/*.parquet')
                     GROUP BY Origin ORDER BY flights DESC LIMIT {TOP_N}""").df()
ap_path = META/"airports.csv"
if not ap_path.exists(): download(AIRPORTS_CSV, ap_path)
oa = pd.read_csv(ap_path, low_memory=False)
oa = oa[oa.iata_code.notna()].sort_values("type", key=lambda s: s.map({"large_airport":0,"medium_airport":1}).fillna(2))
oa = oa.drop_duplicates("iata_code")[["iata_code","name","latitude_deg","longitude_deg"]]
top = top.merge(oa, left_on="iata", right_on="iata_code", how="left").drop(columns="iata_code")
print("airports without coordinates:", top[top.latitude_deg.isna()].iata.tolist() or "none")
top.head(10)

airports without coordinates: none


,iata,flights,name,latitude_deg,longitude_deg
0,ATL,5971746,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101
1,ORD,4831539,Chicago O'Hare International Airport,41.978600,-87.904800
2,DFW,4493443,Dallas Fort Worth International Airport,32.896801,-97.038002
3,DEN,4178538,Denver International Airport,39.860027,-104.673792
4,LAX,3356895,Los Angeles International Airport,33.942501,-118.407997
5,PHX,2861016,Phoenix Sky Harbor International Airport,33.435302,-112.005905
6,CLT,2745871,Charlotte Douglas International Airport,35.214001,-80.943100
7,LAS,2589039,Harry Reid International Airport,36.083361,-115.151817
8,SFO,2481112,San Francisco International Airport,37.619806,-122.374821
9,IAH,2452004,George Bush Intercontinental Airport,29.984400,-95.341400


## Step 2 · Weather stations near each airport

In [ ]:
st_path = META/"ghcnh-station-list.csv"
if not st_path.exists(): download(STATION_LIST, st_path)
raw = pd.read_csv(st_path, header=None, low_memory=False, dtype=str)
if not pd.to_numeric(raw.iloc[0, 1], errors="coerce") == pd.to_numeric(raw.iloc[0, 1], errors="coerce"):
    raw = raw.iloc[1:]                       # first row was a header
st = raw.iloc[:, :3].copy(); st.columns = ["station", "lat", "lon"]
st["name"] = raw.iloc[:, 5] if raw.shape[1] > 5 else ""
st["lat"] = pd.to_numeric(st.lat, errors="coerce"); st["lon"] = pd.to_numeric(st.lon, errors="coerce")
st = st.dropna(subset=["lat","lon"])
print(len(st), "stations; lat range", st.lat.min(), st.lat.max())   # sanity: should be about -90..90
st.head()

38870 stations; lat range -90.0 135.117


,station,lat,lon,name
1,ACL000BARA9,17.5910,-61.8210,BARBUDA
2,ACM00078861,17.1167,-61.7833,COOLIDGE FIELD ANTIGUA (AUX.
3,ACU55-00189,18.6000,-63.4700,SOMBRERO
4,ACW00011647,17.1333,-61.7833,ST JOHNS
5,AEI0000OMAA,24.4330,54.6511,ABU DHABI INTL


In [ ]:
cands = []
for r in top.itertuples():
    if pd.isna(r.latitude_deg): continue
    near = st[(abs(st.lat - r.latitude_deg) < 0.2) & (abs(st.lon - r.longitude_deg) < 0.2)].copy()
    near["km"] = [km(r.latitude_deg, r.longitude_deg, a, b) for a, b in zip(near.lat, near.lon)]
    for c in near[near.km <= MAX_KM].sort_values("km").head(5).itertuples():
        cands.append(dict(iata=r.iata, station=c.station, km=round(c.km, 1), name=c.name))
cands = pd.DataFrame(cands)
print("airports with at least one candidate station:", cands.iata.nunique(), "of", len(top))
print("no candidate:", sorted(set(top.iata) - set(cands.iata)) or "none")
cands.head(15)

airports with at least one candidate station: 50 of 50
no candidate: none


,iata,station,km,name
0,ATL,USW00013874,1.5,ATLANTA HARTSFIELD-JACKSON INT
1,ORD,USW00014810,0.7,PARK RIDGE AF
2,ORD,USW00094846,3.0,CHICAGO OHARE INTL AP
3,DFW,USW00003927,1.5,DAL-FTW WSCMO AP
4,DEN,USW00003017,2.1,DENVER INTL AP
5,LAX,USW00023174,2.0,LOS ANGELES INTL AP
6,PHX,USW00023183,0.9,PHOENIX AP
7,CLT,USW00013881,1.4,CHARLOTTE DOUGLAS AP
8,LAS,USW00023169,1.6,MCCARRAN INTL AP
9,SFO,USW00023234,0.8,SAN FRANCISCO INTL AP


## Step 3 · Keep the candidate whose weather file really exists (checks 2010, 2018, 2024)
This makes ~3 small requests per candidate. Result goes in `AIRPORT_STATION` (you can edit it by hand afterwards).

In [ ]:
probe_rows = []
for r in cands.itertuples():
    ok = [exists(LCD.format(y=y, st=r.station)) for y in PROBE_YEARS]
    probe_rows.append(dict(iata=r.iata, station=r.station, km=r.km, name=r.name, years_found=sum(ok)))
    time.sleep(0.2)
probe = pd.DataFrame(probe_rows)
best = (probe.sort_values(["iata", "years_found", "km"], ascending=[True, False, True])
             .drop_duplicates("iata"))
AIRPORT_STATION = dict(zip(best.iata, best.station))
print(len(AIRPORT_STATION), "airports mapped")
print("weak (found in <3 probe years):", best[best.years_found < 3][["iata","station","years_found"]].values.tolist() or "none")
print("unmapped:", sorted(set(top.iata) - set(AIRPORT_STATION)) or "none")
best.merge(top[["iata","flights"]], on="iata").sort_values("flights", ascending=False).head(15)

50 airports mapped
weak (found in <3 probe years): none
unmapped: none


,iata,station,km,name,years_found,flights
0,ATL,USW00013874,1.5,ATLANTA HARTSFIELD-JACKSON INT,3,5971746
33,ORD,USW00094846,3.0,CHICAGO OHARE INTL AP,3,4831539
12,DFW,USW00003927,1.5,DAL-FTW WSCMO AP,3,4493443
11,DEN,USW00003017,2.1,DENVER INTL AP,3,4178538
23,LAX,USW00023174,2.0,LOS ANGELES INTL AP,3,3356895
36,PHX,USW00023183,0.9,PHOENIX AP,3,2861016
6,CLT,USW00013881,1.4,CHARLOTTE DOUGLAS AP,3,2745871
22,LAS,USW00023169,1.6,MCCARRAN INTL AP,3,2589039
43,SFO,USW00023234,0.8,SAN FRANCISCO INTL AP,3,2481112
19,IAH,USW00012960,1.9,HOUSTON INTERCONTINENTAL AP,3,2452004


**Check the table above:** each airport should map to a station within a few km with years_found = 3. Spot-check a couple of airports you know by their station name.
If one is wrong, override it, e.g. `AIRPORT_STATION["ORD"] = "<station id>"` in the next cell, and re-run Step 4.

In [ ]:
# manual overrides (optional)
# AIRPORT_STATION["ORD"] = "..."
best.to_csv(META/"airport_station_map.csv", index=False)

## Step 4 · Download and convert (per station-year)

In [ ]:
con = duckdb.connect()
failed, log = [], []
for ap, sid in AIRPORT_STATION.items():
    for y in YEARS:
        out = PQ/f"{ap}_{sid}_{y}.parquet"
        csv = RAW/f"{sid}_{y}.csv"
        if out.exists():
            if csv.exists(): csv.unlink()      # clears the leftover from the failed run
            continue
        url = LCD.format(y=y, st=sid)
        try:
            download(url, csv)
            con.execute(f"COPY (SELECT '{ap}' AS airport, * FROM read_csv('{csv}', header=true, all_varchar=true)) "
                        f"TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
            n = con.execute(f"SELECT count(*) FROM read_parquet('{out}')").fetchone()[0]
            log.append(dict(airport=ap, year=y, rows=n)); csv.unlink()
            print(f"[ok  ] {ap} {y}: {n:,} rows")
        except urllib.error.HTTPError as e:
            failed.append((ap, y, f"HTTP {e.code}"))      # e.g. station didn't exist that year
            if csv.exists(): csv.unlink()
        except (urllib.error.URLError, duckdb.Error, OSError) as e:
            failed.append((ap, y, str(e))); print(f"[FAIL] {ap} {y}: {e}")
            if csv.exists(): csv.unlink()
        time.sleep(PAUSE_SECONDS)
print(len(failed), "missing station-years (first 10):", failed[:10])

[FAIL] DCA 2018: Invalid Input Error: Error when sniffing file "/Users/a../Desktop/ADSP32030_Scalable/Project/data/weather/raw/USW00013743_2018.csv".
It was not possible to automatically detect the CSV parsing dialect
The search space used was:
Delimiter Candidates: ',', '|', ';', '	'
Quote/Escape Candidates: ['(no quote)','(no escape)'],['"','(no escape)'],['"','"'],['"','''],['"','\'],[''','(no escape)'],[''','''],[''','"'],[''','\']
Comment Candidates: '\0', '#'
Encoding: utf-8
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Set delimiter (e.g., delim=',')
* Set quote (e.g., quote='"')
* Set escape (e.g., escape='"')
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable ignore errors (ignore_errors=true) to ignore potential errors
* Enable null pa

In [ ]:
ap, y = "DCA", 2018
sid = AIRPORT_STATION[ap]
csv = RAW/f"{sid}_{y}.csv"; out = PQ/f"{ap}_{sid}_{y}.parquet"
download(LCD.format(y=y, st=sid), csv)
print(open(csv, errors="ignore").read(400))
con.execute(f"COPY (SELECT '{ap}' AS airport, * FROM read_csv('{csv}', header=true, all_varchar=true, "
            f"strict_mode=false, ignore_errors=true, null_padding=true)) "
            f"TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(con.execute(f"SELECT count(*) FROM read_parquet('{out}')").fetchone()[0], "rows")
csv.unlink()

"STATION","DATE","LATITUDE","LONGITUDE","ELEVATION","NAME","REPORT_TYPE","SOURCE","HourlyAltimeterSetting","HourlyDewPointTemperature","HourlyDryBulbTemperature","HourlyPrecipitation","HourlyPresentWeatherType","HourlyPressureChange","HourlyPressureTendency","HourlyRelativeHumidity","HourlySkyConditions","HourlySeaLevelPressure","HourlyStationPressure","HourlyVisibility","HourlyWetBulbTemperature"
15615 rows


## Step 5 · Summary (for the slides)

In [ ]:
files = list(PQ.glob("*.parquet"))
print(duckdb.sql(f"SELECT count(*) AS rows, count(DISTINCT airport) AS airports FROM read_parquet('{PQ}/*.parquet')").df() if files else "no files yet")
if files:
    print(f"parquet: {sum(f.stat().st_size for f in files)/1e6:,.0f} MB in {len(files)} station-year files")
    cov = duckdb.sql(f"SELECT airport, count(DISTINCT regexp_extract(filename, '_(\\d{{4}})\\.parquet', 1)) AS years FROM read_parquet('{PQ}/*.parquet', filename=true) GROUP BY 1 ORDER BY years").df()
    print("airports with the fewest years of weather:"); display(cov.head(10))

       rows  airports
0  10660339        50
parquet: 463 MB in 850 station-year files
airports with the fewest years of weather:


,airport,years
0,PHX,17
1,DCA,17
2,OAK,17
3,FLL,17
4,SAT,17
5,LAS,17
6,CLT,17
7,HNL,17
8,SNA,17
9,BWI,17
